<a href="https://colab.research.google.com/github/jayjanii/sentiment-analysis/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
from datasets import load_dataset

In [32]:
dataset = load_dataset('imdb')

In [106]:
dataset['train'][200]

{'text': 'This is an action Western. James Steart leads an all star cast in the scenic Northwest, which is filmed in great splendor. The scenery and costumes are great. There is action and adventure. Stewart plays a wealthy cattleman who runs afoul of a crooked government in the old Nothwest.<br /><br />The main drawback is the stereotypical cynic that Hollywood has always made into a hero. Even when this movie was made, the cynic was the stereotypical hero, and the one Stewart portrays really has few saving graces. He is kind to his two partners, and that does give him an extra dimension of credibility and likability.<br /><br />However, he is so piggish to everyone else, it is hard to really care for him, or to accept him. He is much like the one dimensional spaghetti Western characters (cut not that bad).<br /><br />Still, the minor characters are quite enjoyable. Walter Brennan, Royal Dano, Harry Morgan, and others make this worth watching.',
 'label': 0}

In [109]:
len(dataset['train'])

25000

In [110]:
len(dataset['test'])

25000

In [107]:
from transformers import BasicTokenizer
tokenizer = BasicTokenizer(do_lower_case=True)
tokenizer.tokenize("The movie was AMAZING!!")

['the', 'movie', 'was', 'amazing', '!', '!']

In [36]:
def build_vocabulary(dataset):
    vocab = {"<pad>": 0, "<unk>": 1}
    id = 2
    for data in dataset['train']:
        token_list = tokenizer.tokenize(data['text'])
        for token in token_list:
            if token not in vocab:
                vocab[token] = id
                id += 1

    return vocab

In [37]:
vocab = build_vocabulary(dataset)
print(len(vocab))

74878


In [38]:
import torch

def encode_review(text, vocab, max_length=256):
    tokenized_text = tokenizer.tokenize(text)
    tokenized_text = tokenized_text[:max_length]

    for i, token in enumerate(tokenized_text, start=0):
        if token in vocab:
            tokenized_text[i] = vocab[token] # vocab ID
        else:
            tokenized_text[i] = vocab["<unk>"] # not in vocab

    if len(tokenized_text) < max_length:
        n = max_length - len(tokenized_text)
        tokenized_text.extend([vocab["<pad>"]] * n) # padding

    tensor = torch.tensor(tokenized_text)

    return tensor

In [39]:
tensor = encode_review(dataset['train'][0]['text'], vocab)
print(tensor.shape)
print(tensor[:10])

torch.Size([256])
tensor([ 2,  3,  2,  4,  5,  6,  7,  8,  9, 10])


In [40]:
from torch.utils.data import Dataset

class IMDBDataset(Dataset):
    def __init__(self, data, vocab, max_length=256):
        self.data = []
        self.labels = []
        for i in range(len(data)):
            self.data.append(encode_review(data[i]['text'], vocab, max_length))
            self.labels.append(data[i]['label'])
        self.vocab = vocab
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        encoded_review = self.data[idx]
        review_label = self.labels[idx]
        return (encoded_review, review_label)

In [91]:
from torch.utils.data import DataLoader

max_length = 512

train_loader = DataLoader(IMDBDataset(dataset['train'], vocab, max_length=max_length),
                          batch_size=32,
                          shuffle=True
                          )

test_loader = DataLoader(IMDBDataset(dataset['test'], vocab, max_length=max_length),
                         batch_size=32,
                         shuffle=False
                         )

In [97]:
import torch.nn as nn
import torch.nn.functional as F

class SentimentModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear(embed_dim, embed_dim//2)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(embed_dim//2, 1)

    def forward(self, x):
        x = self.embeddings(x)
        x = torch.mean(x, dim=1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = torch.sigmoid(x)

        return x

In [43]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [50]:
from torch.optim import Adam

In [99]:
def model_eval(SentimentModel):

    model.eval()

    correct_count = 0
    sample_count = 0
    total_loss = 0

    with torch.no_grad():
        for input, label in test_loader:
            input, label = input.to(device), label.to(device=device, dtype=torch.float)
            pred = torch.squeeze(model(input))
            loss = loss_fn(pred, label)
            total_loss += loss.item()

            binary_preds = (pred >= 0.5)
            correct = (binary_preds == label)
            correct_count += correct.sum().item()

            sample_count += len(label)


    accuracy = correct_count / sample_count
    avg_loss = total_loss / len(test_loader)
    print(f'Accuracy: {100*accuracy:.2f}% | Test Loss: {avg_loss}')

    return accuracy

In [100]:
model = SentimentModel(vocab_size=len(vocab), embed_dim=256).to(device)
optim = Adam(model.parameters(), lr=5e-4)
loss_fn = nn.BCELoss()

model.train()
epochs = 10
model_eval(model)

best_accuracy = 0.0

for epoch in range(1, epochs + 1):
    total_loss = 0
    for input, label in train_loader:
        optim.zero_grad()
        input, label = input.to(device), label.to(device=device, dtype=torch.float)
        pred = torch.squeeze(model(input))
        loss = loss_fn(pred, label)
        total_loss += loss.item()
        loss.backward()
        optim.step()
    model_eval(model)
    print(f'Epoch: {epoch} | Train Loss: {total_loss / len(train_loader)}')

accuracy = model_eval(model)

if accuracy > best_accuracy:
    best_accuracy = accuracy
    torch.save(model.state_dict(), 'best_model.pth')


Accuracy: 50.00% | Test Loss: 0.6939085526844425
Accuracy: 66.90% | Test Loss: 0.5820973999893574
Epoch: 1 | Train Loss: 0.6429505489976205
Accuracy: 80.79% | Test Loss: 0.4091754276638903
Epoch: 2 | Train Loss: 0.43073670947185866
Accuracy: 77.70% | Test Loss: 0.4796051688246962
Epoch: 3 | Train Loss: 0.31253926072965194
Accuracy: 87.29% | Test Loss: 0.3090643768467943
Epoch: 4 | Train Loss: 0.25355278268037246
Accuracy: 87.46% | Test Loss: 0.3042472359388495
Epoch: 5 | Train Loss: 0.2169152602191319
Accuracy: 88.23% | Test Loss: 0.294411457970243
Epoch: 6 | Train Loss: 0.18643275131959744
Accuracy: 88.28% | Test Loss: 0.29782041047444885
Epoch: 7 | Train Loss: 0.15905933723310986
Accuracy: 86.72% | Test Loss: 0.3385614699557843
Epoch: 8 | Train Loss: 0.14166902855534078
Accuracy: 81.43% | Test Loss: 0.47868969696252356
Epoch: 9 | Train Loss: 0.11827895134363485
Accuracy: 87.61% | Test Loss: 0.34033192324040035
Epoch: 10 | Train Loss: 0.10902494856435091
Accuracy: 87.61% | Test Loss: 

In [104]:
review = '''I must admit, I still believe "Frank Herbert's Dune" to be unfilmable. I realised that, watching Dune: Part One in my local movie theater today. That's why people who have read the book should see this movie adaptation as a single piece of art. Forget the book while watching this movie. Dune, the book drives on the thoughts and inner emotions of the characters - the things that are left unsaid. Visually, that can only be adapted to a certain extent. Villeneuve has done a better job than anyone could ask for. I think he had to compromise on not bringing even more details to the movie, considering that the majority of the people will not have read the book. The information overload would have been difficult to follow for most. This movie is definitely slow, so if you are accustomed to constant dopamine stimulation then you might find it boring.The visuals in combination with the sound/music are hauntingly beautiful and will stick in your mind for a long time. I got goosebumps almost throughout the entire movie. Dune: Part One definitely serves as a stepping stone for Dune: Part Two, and more to come... If you are looking for action, Dune: Part Two is definitely the movie to be waiting for. So grab some friends and go watch this movie. Make "Dune: Part Two" happen. This is only the beginning.'''
tensor = encode_review(review, vocab)

model = SentimentModel(vocab_size=len(vocab), embed_dim=256).to(device)
model.load_state_dict(torch.load('best_model.pth'))
with torch.no_grad():
    tensor = torch.unsqueeze(tensor, 0)
    tensor = tensor.to(device)
    pred = model(tensor)

print(f'Review: {review}')
print(f'Sentiment: {'Negative' if pred.item() < 0.5 else 'Positive'}')
print(f'Pred: {pred.item()}')

Review: I must admit, I still believe "Frank Herbert's Dune" to be unfilmable. I realised that, watching Dune: Part One in my local movie theater today. That's why people who have read the book should see this movie adaptation as a single piece of art. Forget the book while watching this movie. Dune, the book drives on the thoughts and inner emotions of the characters - the things that are left unsaid. Visually, that can only be adapted to a certain extent. Villeneuve has done a better job than anyone could ask for. I think he had to compromise on not bringing even more details to the movie, considering that the majority of the people will not have read the book. The information overload would have been difficult to follow for most. This movie is definitely slow, so if you are accustomed to constant dopamine stimulation then you might find it boring.The visuals in combination with the sound/music are hauntingly beautiful and will stick in your mind for a long time. I got goosebumps alm